In [1]:
# import 4FGL catalog and save features
import numpy as np
#import astropy
from astropy.io import fits
from matplotlib import pyplot
import pandas as pd
import os
import copy

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression

try:
    import dio3
    import multi_fgl_utils as util
except:
    import sys
    sys.path.append('../lib/')
    import dio3
    import multi_fgl_utils as util

In [2]:
if 1:
    import importlib
    importlib.reload(util);
np.set_printoptions(precision=4)

In [22]:
cat_name = '4FGL-DR4' # '4FGL-DR3', '8year'
if cat_name == '4FGL-DR4':
    baseline_case = '4FGL-DR4_GMM_RF'
    case = '4FGL-DR4_GMM_RF'
elif cat_name == '4FGL-DR3':
    baseline_case = '4FGL-DR3_GMM_RF'
    case = '4FGL-DR3_GMM_RF'
elif cat_name == '4FGL-DR2':
    baseline_case = '4FGL-DR2_GMM_RF'
    case = '4FGL-DR2_GMM_RF'
elif cat_name == '4FGL-DR1':
    baseline_case = '4FGL-DR1_GMM_RF'
    case = '4FGL-DR1_GMM_RF'
elif cat_name == '8year':
    baseline_case = '8year_GMM_RF'
    case = '8year_GMM_RF'

baseline = (case == baseline_case)
root_folder = '../'
config_folder = root_folder + 'config/'

# generic inputs
cdict = util.get_config_dict(cat_name=cat_name, 
                             baseline_case=baseline_case, 
                             case=case, 
                             folder=config_folder)

root_folder = cdict.get('root_folder', '../')
folder_in = root_folder + cdict.get('folder_in', 'data_in/')
folder_out = root_folder + cdict.get('folder_out', 'data_out/')
if not os.path.exists(folder_out):
    os.makedirs(folder_out)
Eind = int(cdict.get('Eind', 1000))
cdict['add_plec_pars'] = 0
add_plec_pars = int(cdict.get('add_plec_pars', 0))
if add_plec_pars:
    plec_keys = ['PLEC_IndexS', 'PLEC_ExpfactorS', 'PLEC_Exp_Index', 'PLEC_SigCurv']
    plec_str = '_add_plec_pars'
else:
    plec_str = ''
    
xdf_fn = util.get_xdf_fn(cdict)
print(xdf_fn)

../data_out/xdf_4FGL-DR4_Eind1000MeV.csv


In [25]:
fn = util.get_cat_fn(cdict)
print(fn)
cat_hdu = fits.open(fn)
ctab = cat_hdu[1].data 
ns = len(ctab)


../data_in/gll_psc_v34.fit


In [26]:
ctab.dtype.names

('Source_Name',
 'DataRelease',
 'RAJ2000',
 'DEJ2000',
 'GLON',
 'GLAT',
 'Conf_68_SemiMajor',
 'Conf_68_SemiMinor',
 'Conf_68_PosAng',
 'Conf_95_SemiMajor',
 'Conf_95_SemiMinor',
 'Conf_95_PosAng',
 'ROI_num',
 'Extended_Source_Name',
 'Signif_Avg',
 'Pivot_Energy',
 'Flux1000',
 'Unc_Flux1000',
 'Energy_Flux100',
 'Unc_Energy_Flux100',
 'SpectrumType',
 'PL_Flux_Density',
 'Unc_PL_Flux_Density',
 'PL_Index',
 'Unc_PL_Index',
 'LP_Flux_Density',
 'Unc_LP_Flux_Density',
 'LP_Index',
 'Unc_LP_Index',
 'LP_beta',
 'Unc_LP_beta',
 'LP_SigCurv',
 'LP_EPeak',
 'Unc_LP_EPeak',
 'PLEC_Flux_Density',
 'Unc_PLEC_Flux_Density',
 'PLEC_IndexS',
 'Unc_PLEC_IndexS',
 'PLEC_ExpfactorS',
 'Unc_PLEC_ExpfactorS',
 'PLEC_Exp_Index',
 'Unc_PLEC_Exp_Index',
 'PLEC_SigCurv',
 'PLEC_EPeak',
 'Unc_PLEC_EPeak',
 'Npred',
 'Flux_Band',
 'Unc_Flux_Band',
 'nuFnu_Band',
 'Sqrt_TS_Band',
 'Variability_Index',
 'Frac_Variability',
 'Unc_Frac_Variability',
 'Signif_Peak',
 'Flux_Peak',
 'Unc_Flux_Peak',
 'Time_Pea

In [27]:
# features_dict
fdict = {}
fdict['sin(GLAT)'] = np.sin(np.deg2rad(ctab['GLAT']))
fdict['cos(GLON)'] = np.cos(np.deg2rad(ctab['GLON']))
fdict['sin(GLON)'] = np.sin(np.deg2rad(ctab['GLON']))
fdict['log10(Energy_Flux100)'] = np.log10(ctab['Energy_Flux100'])
fdict['log10(Unc_Energy_Flux100)'] = np.log10(ctab['Unc_Energy_Flux100'])
fdict['log10(Signif_Avg)'] = np.log10(ctab['Signif_Avg'])
fdict['LP_index%iMeV' % Eind] = util.logpar_index_cat(Eind, ctab)
fdict['LP_beta'] = np.array(ctab['LP_beta']).astype('float')
fdict['LP_SigCurv'] = np.array(ctab['LP_SigCurv']).astype('float')
fdict['log10(Variability_Index)'] = np.log10(ctab['Variability_Index'])
if add_plec_pars:
    for key in plec_keys:
        fdict[key] = ctab[key]
snames = np.array([name.strip() for name in ctab['Source_Name']])
xdf = pd.DataFrame(data=fdict, index=snames)
    #print(len(fdict))

    
nan_mask = np.any(np.isnan(xdf), axis=1)
inf_mask = np.any(np.isinf(xdf), axis=1)
ok_mask = ~(nan_mask | inf_mask)
#print(np.sum(~ok_mask))
print('Problems with features of this source(s):')
print(xdf[~ok_mask])
print('class(es):', ctab['CLASS1'][~ok_mask])
#print((len(xdf), len(ctab), sum(ok_mask)))
# save xdf

class_name = 'CLASS1'
features = xdf.columns
fgl_classes = np.array([cl.lower() for cl in ctab['CLASS1']])
fgl_classes[fgl_classes == ''] = 'unas'
xdf[class_name] = fgl_classes
xdf = xdf[ok_mask]
if 1:
    print()
    print('save dataframe with feaures to file:')
    print(xdf_fn)
    xdf.to_csv(xdf_fn)
    xdf = xdf[features]
#print((len(xdf), len(ctab), sum(ok_mask)))

Problems with features of this source(s):
                    sin(GLAT)  cos(GLON)  sin(GLON)  log10(Energy_Flux100)  \
4FGL J0358.4-5446   -0.724117  -0.080243  -0.996775             -12.825364   
4FGL J0534.5+2201i  -0.100735  -0.996845  -0.079377              -9.388094   
4FGL J1820.8-2822   -0.112528   0.997195   0.074847             -11.528564   
4FGL J2010.2-2523   -0.466539   0.956318   0.292330             -12.165505   

                    log10(Unc_Energy_Flux100)  log10(Signif_Avg)  \
4FGL J0358.4-5446                  -12.533574          -1.608788   
4FGL J0534.5+2201i                        NaN                NaN   
4FGL J1820.8-2822                  -12.150924           0.597425   
4FGL J2010.2-2523                  -12.282506          -0.323367   

                    LP_index1000MeV  LP_beta  LP_SigCurv  \
4FGL J0358.4-5446               NaN      NaN         0.0   
4FGL J0534.5+2201i         1.381586     0.08         0.0   
4FGL J1820.8-2822               NaN      NaN  

/var/folders/mb/yxqk032n4lvgsvnl3lqzbwvr0000gn/T/ipykernel_19936/38320373.py:12: RuntimeWarning: invalid value encountered in log10
  fdict['log10(Variability_Index)'] = np.log10(ctab['Variability_Index'])


In [7]:
xdf['LP_beta']

4FGL J0000.3-7355    0.109999
4FGL J0000.5+0743    0.109660
4FGL J0000.7+2530    0.251122
4FGL J0001.2+4741    0.049796
4FGL J0001.2-0747    0.037899
                       ...   
4FGL J2359.3-2049    0.004414
4FGL J2359.3+1444    0.222978
4FGL J2359.7-5041    0.446554
4FGL J2359.9-3736    0.185941
4FGL J2359.9+3145    0.139681
Name: LP_beta, Length: 7191, dtype: float64

In [8]:
xdf['LP_index1000MeV']

4FGL J0000.3-7355    1.984765
4FGL J0000.5+0743    2.176364
4FGL J0000.7+2530    0.902878
4FGL J0001.2+4741    1.997826
4FGL J0001.2-0747    1.997502
                       ...   
4FGL J2359.3-2049    1.914815
4FGL J2359.3+1444    2.708066
4FGL J2359.7-5041    2.936976
4FGL J2359.9-3736    1.755914
4FGL J2359.9+3145    1.368057
Name: LP_index1000MeV, Length: 7191, dtype: float32

In [9]:
len(ctab['CLASS1'][~ok_mask])

4

In [10]:
ctab['Variability_Index'][~ok_mask][0], ctab['Unc_Energy_Flux100'][~ok_mask][0], ctab['Signif_Avg'][~ok_mask][0], 

(13.053247, 2.92702e-13, 0.024615694)

In [11]:
len(set(fgl_classes))

25

In [12]:
set(fgl_classes)

{'agn',
 'bcu',
 'bin',
 'bll',
 'css',
 'fsrq',
 'gal',
 'gc',
 'glc',
 'hmb',
 'lmb',
 'msp',
 'nlsy1',
 'nov',
 'psr',
 'pwn',
 'rdg',
 'sbg',
 'sey',
 'sfr',
 'snr',
 'spp',
 'ssrq',
 'unas',
 'unk'}

In [13]:
xdf.columns

Index(['sin(GLAT)', 'cos(GLON)', 'sin(GLON)', 'log10(Energy_Flux100)',
       'log10(Unc_Energy_Flux100)', 'log10(Signif_Avg)', 'LP_index1000MeV',
       'LP_beta', 'LP_SigCurv', 'log10(Variability_Index)', 'CLASS1'],
      dtype='object')

In [14]:
mask0 = (xdf['CLASS1'] == 'unas')
mask = mask0 & (np.abs(xdf['sin(GLAT)']) > np.sin(np.deg2rad(10.)))

In [15]:
sum(mask), sum(mask0)

(1282, 2428)